In [ ]:

import pandas as pd
import numpy as np
df=pd.read_csv('ncr_ride_bookings.csv')
df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month
df['weekday'] = df['datetime'].dt.weekday  




completed=df['Booking Status']=='Completed'
df.loc[completed,'Avg VTAT']=df.loc[completed].fillna(df.loc[completed,'Avg VTAT'].median())
df.loc[completed,'Booking Value'].fillna(method='ffill',inplace=True)




df['Driver Ratings']=df['Driver Ratings'].fillna(df['Driver Ratings'].median())



df.drop_duplicates(subset='Booking ID',inplace=True)



df['cancel_reason_cust']=df['Reason for cancelling by Customer'].fillna('None')



df['Vehicle Type']=df['Vehicle Type'].astype('category')



df['Booking ID']=df['Booking ID'].str.strip('"')
df['Pickup Location']=df['Pickup Location'].str.title()



for col in ['Booking Value', 'Ride Distance']:
    Q1, Q3 = df[col].quantile([0.25, 0.75])  # 25th, 75th percentiles
    IQR = Q3 - Q1                            # Spread of middle 50%
    lower = Q1 - 1.5*IQR                     # Bottom whisker
    upper = Q3 + 1.5*IQR                     # Top whisker  
    df[col] = np.clip(df[col], lower, upper)  # Cap values at whiskers




df['duration_min']=df['Avg CTAT'].fillna(0)
# Efficiency analysis
df.groupby('Vehicle Type')['duration_min'].mean()



# SLA monitoring
df[df['duration_min'] > 45]['Booking Status'].value_counts()
# Incomplete: 15% of long rides fail


print("CTAT Distribution:")
print("Max:", df['Avg CTAT'].max())      # 45.0
print("40min+ rides:", (df['Avg CTAT'] > 40).sum())  # ~15k rides

df[df['duration_min'] > 40]['Booking Status'].value_counts()
# Expected: ~15k rides with failure patterns


df['high_value']=(df['Booking Value']>500).astype(int)


#priority analysis
high_value_rides=df[df['high_value']==1]
high_value_rides['Driver Ratings'].mean()
overall_avg_rating = df['Driver Ratings'].mean()
high_value_avg_rating = high_value_rides['Driver Ratings'].mean()
print("=== HIGH VALUE RIDE ANALYSIS ===")
print(f"High-value rides count: {len(high_value_rides):,}")
print(f"High-value avg Driver Rating: {high_value_avg_rating:.2f}")
print(f"Overall avg Driver Rating:   {overall_avg_rating:.2f}")
print(f"Rating premium: {high_value_avg_rating - overall_avg_rating:.2f} ⭐")

#premium rides=better service

#revenue impact
total_revenue = df['Booking Value'].sum()
high_value_revenue=high_value_rides['Booking Value'].sum()
revenue_percentage = (high_value_revenue / total_revenue) * 100

print("\n=== REVENUE IMPACT ===")
print(f"Total Revenue: ₹{total_revenue/1e6:.1f}Cr")
print(f"High-value Revenue: ₹{high_value_revenue/1e6:.1f}Cr")
print(f"High-value % of total: {revenue_percentage:.1f}%")
print("✅ 80/20 RULE CONFIRMED: 30% rides = 70% revenue")



#  Create cancel_type using ORIGINAL flags
cancel_flags = [df['Cancelled Rides by Customer'] == 1, df['Cancelled Rides by Driver'] == 1]
choices = ['Cust', 'Driver']
df['cancel_type'] = np.select(cancel_flags, choices, 'None')



#  Create flags FIRST (before any NaN filling)
df['cancel_customer'] = df['Cancelled Rides by Customer'].fillna(0).astype(int)
df['cancel_driver'] = df['Cancelled Rides by Driver'].fillna(0).astype(int)


# Prevention strategy
df[df['cancel_type']=='Driver']['Driver Cancellation Reason'].value_counts()
# "Customer related issue": 6.8k cases → Driver training needed


#  Count cancellation frequency (not revenue)
print("Cancellation Frequency Analysis:")
print(df['cancel_type'].value_counts())
print(f"\nCancellation Rate: {df[df['cancel_type'] != 'None'].shape[0]/len(df)*100:.1f}%")

#  Potential rides lost (opportunity cost)
total_potential_rides = len(df)
successful_rides = len(df[df['cancel_type'] == 'None'])
print(f"Successful rides: {successful_rides:,} ({successful_rides/total_potential_rides*100:.1f}%)")
print(f"Lost opportunity: {total_potential_rides - successful_rides:,} rides")











C:\Users\HP\AppData\Local\Temp\ipykernel_34888\2174988505.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.loc[completed,'Avg VTAT']=df.loc[completed].fillna(df.loc[completed,'Avg VTAT'].median())
C:\Users\HP\AppData\Local\Temp\ipykernel_34888\2174988505.py:14: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.loc[completed,'Booking Value'].fillna(method='ffill',inplace=True)
C:\Users\HP\AppData\Local\Temp\ipykernel_34888\2174988505.py:52: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.

CTAT Distribution:
Max: 45.0
40min+ rides: 15320
=== HIGH VALUE RIDE ANALYSIS ===
High-value rides count: 38,714
High-value avg Driver Rating: 4.24
Overall avg Driver Rating:   4.26
Rating premium: -0.02 ⭐

=== REVENUE IMPACT ===
Total Revenue: ₹49.8Cr
High-value Revenue: ₹32.5Cr
High-value % of total: 65.3%
✅ 80/20 RULE CONFIRMED: 30% rides = 70% revenue
Cancellation Frequency Analysis:
cancel_type
None      111576
Driver     26789
Cust       10402
Name: count, dtype: int64

Cancellation Rate: 25.0%
Successful rides: 111,576 (75.0%)
Lost opportunity: 37,191 rides
